# 02 — Match Outcome Model

Logistic Regression baseline, XGBoost production model, isotonic calibration, and 2022 WC held-out evaluation.

In [ ]:
import sys
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, brier_score_loss, confusion_matrix, log_loss
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

PROJECT_ROOT = Path("..").resolve()
sys.path.insert(0, str(PROJECT_ROOT))
from src.feature_engineering import FEATURE_COLUMNS

In [ ]:
df = pd.read_parquet(PROJECT_ROOT / "data/processed/match_features.parquet")
df["match_date"] = pd.to_datetime(df["match_date"])
feature_cols = [
    c for c in FEATURE_COLUMNS if c in df.columns
    and df[c].notna().any() and df[c].nunique(dropna=True) > 1
]

is_mens_wc = df["competition"].str.contains("FIFA World Cup", case=False, na=False)
is_wc_2018 = (df["match_date"] >= "2018-06-01") & (df["match_date"] <= "2018-07-31") & is_mens_wc
is_wc_2022 = (df["match_date"] >= "2022-11-01") & (df["match_date"] <= "2022-12-31") & is_mens_wc
train = df[~is_wc_2018 & ~is_wc_2022]
val = df[is_wc_2018]
test = df[is_wc_2022]
print(f"Train: {len(train)}, Val: {len(val)}, Test: {len(test)}")
print(f"Features used: {len(feature_cols)}")

In [ ]:
X_train, y_train = train[feature_cols], train["outcome"]
X_val, y_val = val[feature_cols], val["outcome"]
X_test, y_test = test[feature_cols], test["outcome"]

lr = Pipeline([("scaler", StandardScaler()), ("model", LogisticRegression(multi_class="multinomial", max_iter=2000))])
lr.fit(X_train, y_train)
print("LR val log-loss:", log_loss(y_val, lr.predict_proba(X_val), labels=[0,1,2]))

In [ ]:
xgb_pipe = Pipeline([("scaler", StandardScaler()), ("model", XGBClassifier(objective="multi:softprob", num_class=3, eval_metric="mlogloss", random_state=42, use_label_encoder=False))])
grid = GridSearchCV(xgb_pipe, {"model__max_depth": [3,5], "model__learning_rate": [0.05, 0.1], "model__n_estimators": [100, 200]}, cv=3, scoring="neg_log_loss", n_jobs=-1)
grid.fit(X_train, y_train)
best = grid.best_estimator_
print("Best params:", grid.best_params_)
print("XGB val log-loss:", log_loss(y_val, best.predict_proba(X_val), labels=[0,1,2]))

In [ ]:
calibrated = CalibratedClassifierCV(best, method="isotonic", cv="prefit")
calibrated.fit(X_val, y_val)

test_proba = calibrated.predict_proba(X_test)
print("2022 WC Test:")
print("  log-loss:", log_loss(y_test, test_proba, labels=[0,1,2]))
print("  accuracy:", accuracy_score(y_test, test_proba.argmax(axis=1)))
print("\nConfusion matrix:\n", confusion_matrix(y_test, test_proba.argmax(axis=1)))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
labels = ["Loss", "Draw", "Win"]
for c, ax in enumerate(axes):
    raw_p = best.predict_proba(X_val)[:, c]
    cal_p = calibrated.predict_proba(X_val)[:, c]
    y_bin = (y_val == c).astype(int)
    ax.plot(*calibration_curve(y_bin, raw_p, n_bins=8), marker="o", label="Raw XGB")
    ax.plot(*calibration_curve(y_bin, cal_p, n_bins=8), marker="s", label="Calibrated")
    ax.plot([0,1],[0,1],"k--", alpha=0.5)
    ax.set_title(labels[c]); ax.legend()
plt.suptitle("Calibration Curves (2018 WC Validation)"); plt.tight_layout()
plt.savefig(PROJECT_ROOT / "docs/images/calibration_curves.png", dpi=150)
plt.show()